# Test de positionnement — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/positionnement/03_solutions.ipynb)

Les réponses enregistrées (cellules marquées `answer`, lues par `tools/build_answers.py`), puis les mêmes vérifications que le notebook de l'apprenant, remplies avec les bonnes réponses. Les démarches sont dans `03_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("ref")  # reference implementation

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les outils

Exécute cette cellule (« Tout exécuter » le fait pour toi). Mets `FINISHED = True` quand tu as reporté toutes tes réponses.

In [2]:
# Tools of the placement test: they store each check, then turn the results into your grid.
from fractions import Fraction

FINISHED = True   # put True when you have reported ALL your answers (a question left at ... then counts as missed)
RESULTS = {}       # filled by the check cells below: exercise -> [(right?, status)] for its questions

GRID = {"R1": ("Nombres et calcul", ["POS.1", "POS.2"]),
        "R2": ("Calcul littéral et équations", ["POS.3", "POS.4"]),
        "R3": ("Fonctions", ["POS.5", "POS.6"]),
        "R4": ("Logique, ensembles, Σ et Π", ["POS.7", "POS.8"]),
        "R5": ("Suites", ["POS.9", "POS.10"]),
        "R6": ("Exponentielle et logarithmes", ["POS.11", "POS.12"]),
        "R7": ("Limites et dérivation", ["POS.13", "POS.14"]),
        "R8": ("Probabilités de base", ["POS.15", "POS.16"]),
        "R9": ("Géométrie du plan, vecteurs et trigonométrie", ["POS.17", "POS.18"])}
ADVICE = {2: "tu peux sauter", 1: "à survoler", 0: "à faire en entier"}


def is_form_problem(result):
    """A message about the way the answer is written (quotes, comma, fraction typed as a decimal), not its value."""
    return result.status == "type" or (result.status == "near" and "valeur approchée" in result.message)


def check_all(ex_id, letters, answers):
    """Check every question of a placement exercise and return (right?, status) for each.

    Until FINISHED is True, a wrong value only shows ❌: its message could guide a correction, and the test
    measures what you knew on paper. A problem of form (quotes, decimal point...) is always explained.
    """
    results = []
    for letter, answer in zip(letters, answers):
        result = wbm.check(f"{ex_id}{letter}", answer, quiet=True)
        if result.status == "pending":
            line = "— laissée vide : comptée fausse" if FINISHED else "⏳ vide (laisse `...` si tu n'as pas su la faire)"
        elif result.passed:
            line = "✅"
        elif is_form_problem(result):
            line = f"⚠️ la forme, pas la valeur : {result.message} Corrige l'écriture, garde ta valeur."
        elif FINISHED:
            line = f"❌ {result.message}"
        else:
            line = "❌ (le message s'affichera quand FINISHED = True)"
        print(f"{ex_id}{letter} {line}")
        results.append((bool(result.passed), result.status))
    return results


def show_grid(results, finished):
    """Print the verdict of each chapter: an exercise counts when ALL its questions are right."""
    for cid, (title, exercises) in GRID.items():
        checks = [results.get(ex, [(False, "pending")]) for ex in exercises]
        if not finished and any(status == "pending" for c in checks for _, status in c):
            verdict = "⏳ pas encore rempli (ou mets FINISHED = True dans la cellule des outils)"
        else:
            score = sum(all(right for right, _ in c) for c in checks)
            verdict = f"{score}/2 → {ADVICE[score]}"
        print(f"{cid} · {title:<46} {verdict}")
    print("\nCe que veut dire chaque verdict, et ce qu'il faut faire quand même : positionnement/05_grille.md")

## R1 · Nombres et calcul

**POS.1 — Calculer avec des priorités et des fractions** · a) un nombre · b) une fraction irréductible, entre guillemets · c) une fraction irréductible, entre guillemets · d) une fraction irréductible, entre guillemets

In [3]:
wbm.record("POS.1a", 7 - 3 * (2 - 5), mistakes={"la multiplication passe avant la soustraction": -12, "moins par moins donne plus": -2})
wbm.record("POS.1b", Fraction(2, 3) + Fraction(5, 4), kind="fraction", simplified=True, mistakes={"on n'additionne pas les numérateurs et les dénominateurs entre eux": "1"})
wbm.record("POS.1c", Fraction(3, 5) / Fraction(9, 10), kind="fraction", simplified=True, mistakes={"diviser par une fraction, c'est multiplier par son inverse": "27/50"})
wbm.record("POS.1d", (1 - Fraction(1, 4)) / 2, kind="fraction", simplified=True, mistakes={"diviser par 2 rend le nombre plus petit : c'est le dénominateur qui double": "3/2"})

WBM_ANSWER {"hash": "63bc6fc5ebd30f2261bce655a7135746d653124b0bbaeb2093d2ceb03f94d451", "id": "POS.1a", "kind": "int", "magnitude": 1, "mistakes": {"38f88c37e61f4ba32c48947cffcc94fe6b630739086e9810b00906b46b2c6ede": "la multiplication passe avant la soustraction", "89d4b00683cfd838f81da0e03d4976bda6c289a5ff9f74422b9824bcaa53cbe1": "moins par moins donne plus"}, "sign": 1}
📝 Ex POS.1a : réponse enregistrée (int).
WBM_ANSWER {"hash": "cc05e1fd64c164a3bd88e7803a793b8c8e179e2760b3f311f702576290896cad", "id": "POS.1b", "kind": "fraction", "mistakes": {"58307ee06a6a70e01f4d31429a5e4c007a74ca778a40e46197047370500f6f4c": "on n'additionne pas les numérateurs et les dénominateurs entre eux"}, "sign": 1, "simplified": true}
📝 Ex POS.1b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "583c288194dcb2eff6eb998eaced393213c00b3b57aad92a6e4d19f6014d72b6", "id": "POS.1c", "kind": "fraction", "mistakes": {"c564dc4a1727db90057a0ff1974e0321d664494cc903a2d1dde1665f30c45315": "diviser par une fraction,

In [4]:
answer_POS_1a = 7 - 3 * (2 - 5)
answer_POS_1b = Fraction(2, 3) + Fraction(5, 4)
answer_POS_1c = Fraction(3, 5) / Fraction(9, 10)
answer_POS_1d = (1 - Fraction(1, 4)) / 2

RESULTS["POS.1"] = check_all("POS.1", "abcd", [answer_POS_1a, answer_POS_1b, answer_POS_1c, answer_POS_1d])

POS.1a ✅
POS.1b ✅
POS.1c ✅
POS.1d ✅


**POS.2 — Puissances, racines et notation scientifique** · a) un nombre · b) un entier · c) un nombre · d) un décimal, avec un point

In [5]:
wbm.record("POS.2a", 3 ** (6 - 2 - 3), mistakes={"un exposant négatif compte en moins : additionne les exposants avec leur signe": 243})
wbm.record("POS.2b", 6, mistakes={"écris 72 comme un carré parfait multiplié par 2": 36})
wbm.record("POS.2c", round(6e5 / 3e-2), mistakes={"diviser par 10 puissance -2, c'est multiplier par 10 puissance 2": 2000})
wbm.record("POS.2d", 2.5e-3, decimals=5, mistakes={"10 puissance -3 est un millième : la virgule recule de 3 rangs, pas de 4": 0.00025, "10 puissance -3 est un millième : la virgule recule de 3 rangs, pas de 2": 0.025})

WBM_ANSWER {"hash": "a0ef8c8e1c5168f8a54dddb5fdb14fe4e1a0cd4b405b9cc75dd430eaa4fc3cf6", "id": "POS.2a", "kind": "int", "magnitude": 0, "mistakes": {"d583936b41ed0bd730e65388c3ea7d15d026c2f799fc30461ff85bbae5d709e1": "un exposant négatif compte en moins : additionne les exposants avec leur signe"}, "sign": 1}
📝 Ex POS.2a : réponse enregistrée (int).
WBM_ANSWER {"hash": "a49957a89140244fc66283937511c6565865a0b52fd74cd3465e696b1f2f8987", "id": "POS.2b", "kind": "int", "magnitude": 0, "mistakes": {"4a9a43d7a160d1cf874be462650dfaed3aadf47a3506a451f64a4bddab50afe6": "écris 72 comme un carré parfait multiplié par 2"}, "sign": 1}
📝 Ex POS.2b : réponse enregistrée (int).
WBM_ANSWER {"hash": "3eb483042cb999fb7ff8355d1775c82ecd05cf5973fd5e86ca7b64a60dc8248d", "id": "POS.2c", "kind": "int", "magnitude": 7, "mistakes": {"f7d85bca98ac02f55204f58257aa96d2bb5256f75ad670603ee7476bc9818672": "diviser par 10 puissance -2, c'est multiplier par 10 puissance 2"}, "sign": 1}
📝 Ex POS.2c : réponse enregistrée

In [6]:
answer_POS_2a = 3 ** (6 - 2 - 3)
answer_POS_2b = 6
answer_POS_2c = round(6e5 / 3e-2)
answer_POS_2d = 2.5e-3

RESULTS["POS.2"] = check_all("POS.2", "abcd", [answer_POS_2a, answer_POS_2b, answer_POS_2c, answer_POS_2d])

POS.2a ✅
POS.2b ✅
POS.2c ✅
POS.2d ✅


## R2 · Calcul littéral et équations

**POS.3 — Développer, factoriser, résoudre une équation et une inéquation** · a) la liste `[a, b, c]` · b) un nombre · c) un nombre · d) un intervalle, entre guillemets

In [7]:
wbm.record("POS.3a", [2, 5, -12], mistakes={"le terme constant est le produit des deux nombres, avec son signe": [2, 5, 12], "réduis les termes en x avec leurs signes": [2, 11, -12], "développe chaque terme de la première parenthèse par chaque terme de la seconde": [2, 0, -12], "l'ordre demandé est [a, b, c], du terme en x² au terme constant": [-12, 5, 2]})
wbm.record("POS.3b", 5, mistakes={"(x - a)² = x² - 2ax + a² : identifie le double produit": 10, "le signe de a se lit dans le double produit": -5})
wbm.record("POS.3c", 5)
wbm.record("POS.3d", "]-oo ; 2]", kind="interval", mistakes={"diviser par un nombre négatif retourne l'inégalité": "[2 ; +oo[", "l'inégalité est large : la borne est solution": "]-oo ; 2["})

WBM_ANSWER {"decimals": 0, "element_hashes": ["1b52bd4c1a6bfee3989a54a96cc143f45df3f597232f1e4a0918f6d964f0229e", "4997f3399f59f3c0db5aaba56f9787cbb075a2f959f7453268faa7555fad5088", "beeae56b4c7b7d3c743d0843bddbb2ff02a560e5d25932917525d8e520f59d2d"], "hash": "f59aa515015c995c34e801922629d0264992d231434c34c4dbd9c16bd6dd1619", "id": "POS.3a", "integer": true, "kind": "array", "mistakes": {"43568a7f123867fa73818a27e2100b2d1e1e3a27e6eeb1f5e78eb57b1978b9bc": "réduis les termes en x avec leurs signes", "54b3c059ca601e07952adac1690fa31152db5df6fbf515e16293c4376f824d60": "développe chaque terme de la première parenthèse par chaque terme de la seconde", "560bd0497f57207a8a3f036e31acbbf7210c1a2cc9d2ee0bdbc5083eebef95c5": "le terme constant est le produit des deux nombres, avec son signe", "67e7b576bae943188aba1006f881b9228e5fe1433eec2e73168e3a4bc8612e07": "l'ordre demandé est [a, b, c], du terme en x² au terme constant"}, "shape": [3]}
📝 Ex POS.3a : réponse enregistrée (array, 0 décimale(s)).
WB

In [8]:
answer_POS_3a = [2, 5, -12]
answer_POS_3b = 5
answer_POS_3c = 5
answer_POS_3d = "]-oo ; 2]"

RESULTS["POS.3"] = check_all("POS.3", "abcd", [answer_POS_3a, answer_POS_3b, answer_POS_3c, answer_POS_3d])

POS.3a ✅
POS.3b ✅
POS.3c ✅
POS.3d ✅


**POS.4 — Un système de deux équations à deux inconnues** · a) la liste `[x, y]` · b) le numéro 1, 2 ou 3

In [9]:
wbm.record("POS.4a", [3, 2], mistakes={"l'ordre demandé est [x, y]": [2, 3]})
wbm.record("POS.4b", 1, choices=[1, 2, 3], mistakes={"divise la seconde équation par 2 et compare-la à la première": 2, "les deux équations se ressemblent, mais leurs membres de droite sont-ils compatibles ?": 3})

WBM_ANSWER {"decimals": 0, "element_hashes": ["ab5c7b96c7ae5629bfa411b6be9ad8d8eec6b1da8779d5ac2d678c6a48794461", "231ac5084c7556249f02ee8bd2d19bba6ac83bf541790c7103d6e9ba9a0725b4"], "hash": "18b6909426adf034c055c7a2f04a50ef350f64873c5f3263390e0d60f707a695", "id": "POS.4a", "integer": true, "kind": "array", "mistakes": {"717beef50bfb06816c8b7e3af28a346a4da3cb644254f5053a71d6652ec37863": "l'ordre demandé est [x, y]"}, "shape": [2]}
📝 Ex POS.4a : réponse enregistrée (array, 0 décimale(s)).
WBM_ANSWER {"choices": ["1", "2", "3"], "hash": "12003a35fcd6fba070f2b1180923c83a2eaacc23960b5806ba1dd91b5cd638cd", "id": "POS.4b", "kind": "int", "magnitude": 0, "mistakes": {"5d7073b8a447958c6d10eb95b691b752be7e8fcf30abb36783ae306e3283b7dc": "les deux équations se ressemblent, mais leurs membres de droite sont-ils compatibles ?", "b13ac6cf514a1adba8e1a5ab57879f659739cec662ab796c3c93251c244c3437": "divise la seconde équation par 2 et compare-la à la première"}, "sign": 1}
📝 Ex POS.4b : réponse enregis

In [10]:
answer_POS_4a = [3, 2]
answer_POS_4b = 1

RESULTS["POS.4"] = check_all("POS.4", "ab", [answer_POS_4a, answer_POS_4b])

POS.4a ✅
POS.4b ✅


## R3 · Fonctions

**POS.5 — Lire un graphique de fonction** · a) un nombre · b) un nombre · c) un intervalle, entre guillemets · d) un nombre

In [11]:
wbm.record("POS.5a", 1)
wbm.record("POS.5b", 3, mistakes={"regarde aussi les extrémités de la courbe": 2})
wbm.record("POS.5c", "[-1 ; 2]", kind="interval", mistakes={"décroissante : la courbe descend quand x augmente": "[2 ; 4]", "le plus grand intervalle contient ses bornes : crochets fermés": "]-1 ; 2["})
wbm.record("POS.5d", 3, mistakes={"le maximum est une valeur de g(x), pas l'abscisse où elle est atteinte": -1})

WBM_ANSWER {"hash": "15fb1ca01ff4d39feaf4e7899f32ee53c5c8340896bbb0396802081d3060abdc", "id": "POS.5a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex POS.5a : réponse enregistrée (int).
WBM_ANSWER {"hash": "b11b4bd085d2d9cf22b560fe7e4a085f7039db5e2b874643899e007ffcd20f1e", "id": "POS.5b", "kind": "int", "magnitude": 0, "mistakes": {"b0458e78659b46aa98bf1666d640d2a877b9eb97bf9c5d1aa8b5855b88c8f407": "regarde aussi les extrémités de la courbe"}, "sign": 1}
📝 Ex POS.5b : réponse enregistrée (int).
WBM_ANSWER {"hash": "71ac0fcb619c5e42cc186136223eb7466a20dcce9649d17b11e92450099eb36a", "id": "POS.5c", "kind": "interval", "mistakes": {"636deed77056867843cda5e480b8a8a39314531c065c51c3b2e60d9c54c69198": "décroissante : la courbe descend quand x augmente", "7228ced72da7d341ce0734361c219e42ae5e4117d14b292909d653ca1cd36ca1": "le plus grand intervalle contient ses bornes : crochets fermés"}}
📝 Ex POS.5c : réponse enregistrée (interval).
WBM_ANSWER {"hash": "2aea84af81ac2e18f1a7a75edfadcf0fce65f18

In [12]:
answer_POS_5a = 1
answer_POS_5b = 3
answer_POS_5c = "[-1 ; 2]"
answer_POS_5d = 3

RESULTS["POS.5"] = check_all("POS.5", "abcd", [answer_POS_5a, answer_POS_5b, answer_POS_5c, answer_POS_5d])

POS.5a ✅
POS.5b ✅
POS.5c ✅
POS.5d ✅


**POS.6 — Racines et sommet d'un polynôme du second degré** · a) un nombre · b) une liste de nombres · c) une fraction (entre guillemets) ou un décimal · d) une fraction (entre guillemets) ou un décimal

In [13]:
wbm.record("POS.6a", 2 ** 2 - 4 * 2 * (-12), mistakes={"c est négatif : le terme -4ac est donc positif": -92})
wbm.record("POS.6b", [-3, 2], ordered=False, mistakes={"vérifie chaque racine en la remplaçant dans p(x)": [3, -2], "le dénominateur de la formule des racines est 2a": [4, -6]})
wbm.record("POS.6c", "-1/2", kind="fraction", mistakes={"l'abscisse du sommet vaut -b/(2a), avec le signe moins": "1/2"})
wbm.record("POS.6d", "-25/2", kind="fraction", mistakes={"n'oublie pas le coefficient devant x² en calculant p": "-25/4"})

WBM_ANSWER {"hash": "a0f1879247a896dc965fbe8a466a81bf17471e9b72c84ee989d4f1373e9626de", "id": "POS.6a", "kind": "int", "magnitude": 2, "mistakes": {"e1d3953bd6c563e6df1d38504d3fc498b0c272408c2417bb402efef4d0a4621c": "c est négatif : le terme -4ac est donc positif"}, "sign": 1}
📝 Ex POS.6a : réponse enregistrée (int).
WBM_ANSWER {"hash": "346d2ea90f1ce4482c40c0dc75d6e3e6f9bf8700babbfebea7df35e884e3ae8a", "id": "POS.6b", "kind": "set", "mistakes": {"065cbf5216864de8e2cd9fd60bd6ce343b28031b4ae1f1b2c43704401e8da9e5": "vérifie chaque racine en la remplaçant dans p(x)", "607a0ae30ef9055d3a0ceccfdf54c198d7b76b50f6fa97a224273a7671d503a6": "le dénominateur de la formule des racines est 2a"}}
📝 Ex POS.6b : réponse enregistrée (set).
WBM_ANSWER {"hash": "434389687a357efcc86b141ffe2f9dc7c2d1deb239dd65e1a810aecc11db4c74", "id": "POS.6c", "kind": "fraction", "mistakes": {"fa80bd6b47adfbcc60a622bf8b813d070cfd172600ef4f2acfe96359f16bd09f": "l'abscisse du sommet vaut -b/(2a), avec le signe moins"}, "si

In [14]:
answer_POS_6a = 2 ** 2 - 4 * 2 * (-12)
answer_POS_6b = [-3, 2]
answer_POS_6c = "-1/2"
answer_POS_6d = "-25/2"

RESULTS["POS.6"] = check_all("POS.6", "abcd", [answer_POS_6a, answer_POS_6b, answer_POS_6c, answer_POS_6d])

POS.6a ✅
POS.6b ✅
POS.6c ✅
POS.6d ✅


## R4 · Logique, ensembles, Σ et Π

**POS.7 — Sommes avec Σ, somme double et produit avec Π** · a) un nombre · b) un nombre · c) un nombre · d) un nombre

In [15]:
wbm.record("POS.7a", sum(2 * k + 1 for k in range(1, 6)), mistakes={"le +1 s'ajoute à chacun des termes": 31})
wbm.record("POS.7b", sum(10 ** i for i in range(4)), mistakes={"i commence à 0 : le premier terme compte aussi": 1110})
wbm.record("POS.7c", sum(i * j for i in range(1, 3) for j in range(1, 4)))
wbm.record("POS.7d", 2 * 4 * 6, mistakes={"Π multiplie les termes, Σ les additionne": 12})

WBM_ANSWER {"hash": "b7abe0d93930f6e8140f5ed8b9110e4b66bef4a4ca044c0eeac3ce5264d90309", "id": "POS.7a", "kind": "int", "magnitude": 1, "mistakes": {"01722835083b857061be52b5c1681716630956bfcdbf2853ccecd720657de3bb": "le +1 s'ajoute à chacun des termes"}, "sign": 1}
📝 Ex POS.7a : réponse enregistrée (int).
WBM_ANSWER {"hash": "d4209e0b51b29f4c90dc022dfa71e4ff895d245fb9f32a2585268fc6c6a75370", "id": "POS.7b", "kind": "int", "magnitude": 3, "mistakes": {"91a4bd6ae2ddcac350528920469ad9752bc68eac994819fe864427e6b1c9c5c4": "i commence à 0 : le premier terme compte aussi"}, "sign": 1}
📝 Ex POS.7b : réponse enregistrée (int).
WBM_ANSWER {"hash": "b0ab949e0a538751828340de5e3dd0e6286793f187ec50e04988e8da0c19dbbb", "id": "POS.7c", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex POS.7c : réponse enregistrée (int).
WBM_ANSWER {"hash": "f422bf1c72642a8254321d9f0e6d45d3088f051e1eaac358688a615b5ee0dee6", "id": "POS.7d", "kind": "int", "magnitude": 1, "mistakes": {"2d9de95cd0d46bf4e4d727cd5997213cc3dc6e

In [16]:
answer_POS_7a = sum(2 * k + 1 for k in range(1, 6))
answer_POS_7b = sum(10 ** i for i in range(4))
answer_POS_7c = sum(i * j for i in range(1, 3) for j in range(1, 4))
answer_POS_7d = 2 * 4 * 6

RESULTS["POS.7"] = check_all("POS.7", "abcd", [answer_POS_7a, answer_POS_7b, answer_POS_7c, answer_POS_7d])

POS.7a ✅
POS.7b ✅
POS.7c ✅
POS.7d ✅


**POS.8 — La négation d'une phrase avec « pour tout »** · a) le numéro 1, 2, 3 ou 4 · b) `True` ou `False`

In [17]:
wbm.record("POS.8a", 2, choices=[1, 2, 3, 4], mistakes={"la négation de « pour tout » est « il existe »": 1, "« il existe » est juste, mais la conclusion aussi doit être niée": 3, "« aucun » nie trop : il suffit d'un seul exemple pour contredire « pour tout »": 4})
wbm.record("POS.8b", False, mistakes={"un seul contre-exemple suffit à rendre la phrase fausse : cherche-le": True})

WBM_ANSWER {"choices": ["1", "2", "3", "4"], "hash": "8ad431d9d77350996c80557ec43381c6e82d83ff5a56f153e5d0f010b06c5f2f", "id": "POS.8a", "kind": "int", "magnitude": 0, "mistakes": {"005810cd4f7a0951ad69bf667129e8286c05355a9b28d493f014a85144b6f87d": "la négation de « pour tout » est « il existe »", "b56d713f023a071fbac909279fc0cc09f97350656f82db06b26910cd05377ae8": "« il existe » est juste, mais la conclusion aussi doit être niée", "d1df39f7d49bdfa183ff546bfd702b3916c4ec910782bb5fc12582a0ace48485": "« aucun » nie trop : il suffit d'un seul exemple pour contredire « pour tout »"}, "sign": 1}
📝 Ex POS.8a : réponse enregistrée (int).
WBM_ANSWER {"hash": "b8b27828e8784b3be5b3d35a830c035f96f0997cbf9b54dd6e8fc16a9c0d2e89", "id": "POS.8b", "kind": "bool", "mistakes": {"15a86436443f793f405fe61a1ee7aab56ed1789dafc0f66f4686e4eb0a1a396f": "un seul contre-exemple suffit à rendre la phrase fausse : cherche-le"}}
📝 Ex POS.8b : réponse enregistrée (bool).


In [18]:
answer_POS_8a = 2
answer_POS_8b = False

RESULTS["POS.8"] = check_all("POS.8", "ab", [answer_POS_8a, answer_POS_8b])

POS.8a ✅
POS.8b ✅


## R5 · Suites

**POS.9 — Terme et somme d'une suite géométrique** · a) un nombre · b) un nombre · c) une fraction, entre guillemets

In [19]:
wbm.record("POS.9a", 3 * 2 ** 5, mistakes={"u_n = u_0 × q^n : compte bien l'exposant": 192, "u_5 est le 6e terme, car on part de u_0": 48})
wbm.record("POS.9b", 3 * (2 ** 6 - 1), mistakes={"compte les termes de u_0 à u_5 : il y en a un de plus que 5": 93})
wbm.record("POS.9c", Fraction(1, 1) / (1 - Fraction(1, 3)), kind="fraction", mistakes={"la somme infinie vaut 1 / (1 - q), avec un signe moins": "3/4"})

WBM_ANSWER {"hash": "2dbe8acccd9bfb4a07c6dd598cf2e8ff107b6709ba36d7e85ffb0de35b6c7584", "id": "POS.9a", "kind": "int", "magnitude": 1, "mistakes": {"0a1af4136048c45a3e61ecf41d12947d0b1b2b93508cf370fd0e0db5566b0daf": "u_5 est le 6e terme, car on part de u_0", "16392155f0d4d757883058637ce6a2ecc2723b691128cd9dd31f73216c89e894": "u_n = u_0 × q^n : compte bien l'exposant"}, "sign": 1}
📝 Ex POS.9a : réponse enregistrée (int).
WBM_ANSWER {"hash": "891891e50affdf805c7be3d7b18296e36dc5a61fe3d3863cb3dedcb376b810e5", "id": "POS.9b", "kind": "int", "magnitude": 2, "mistakes": {"bcd27a50d51631f602cd666c8c4178c5363f37b157736ae4e7ab011883f7ac1c": "compte les termes de u_0 à u_5 : il y en a un de plus que 5"}, "sign": 1}
📝 Ex POS.9b : réponse enregistrée (int).
WBM_ANSWER {"hash": "15f9842bcdc2df6af440496272b2f4aacbf0447613a3a65ad70cd41dc7cdbae1", "id": "POS.9c", "kind": "fraction", "mistakes": {"b409bf6f101119c4439844c86baf6a19164b456d2ef92b1e2163d5dee58fcfde": "la somme infinie vaut 1 / (1 - q), ave

In [20]:
answer_POS_9a = 3 * 2 ** 5
answer_POS_9b = 3 * (2 ** 6 - 1)
answer_POS_9c = Fraction(1, 1) / (1 - Fraction(1, 3))

RESULTS["POS.9"] = check_all("POS.9", "abc", [answer_POS_9a, answer_POS_9b, answer_POS_9c])

POS.9a ✅
POS.9b ✅
POS.9c ✅


**POS.10 — Une suite définie par récurrence : premiers termes et limite** · a) la liste `[v1, v2, v3]` · b) un nombre

In [21]:
wbm.record("POS.10a", [4, 5, 5.5], decimals=1, mistakes={"chaque terme se calcule à partir du précédent : moitié, puis plus 3": [7, 17, 37], "on demande v1, v2 et v3 : v0 n'en fait pas partie": [2, 4, 5]})
wbm.record("POS.10b", 6, mistakes={"la limite vérifie la même relation que les termes de la suite": 3})

WBM_ANSWER {"decimals": 1, "element_hashes": ["83a6244c4c50a972873b94aba199c9ffd94f37fb228d968d986a9a9f70af0826", "3c5d33359ef01915c443dd52ac6e3fb521044638d6b8e0a3ec07e99edd369b7f", "b7187cb97ad19322d365f16b58854438ecc130f875dacc359089578831d66217"], "hash": "94703a70ec1e8a5c03c0cbe4758a6162131bde00ecb3679938819419c9d0ab2e", "id": "POS.10a", "kind": "array", "mistakes": {"0cbcfe9bb09752a393df41ae497d63182af53334584eb4a2a89c4fd266d2ff86": "on demande v1, v2 et v3 : v0 n'en fait pas partie", "a732269a53b3668cdee5aee1bfa2fad40c5328660c110ba381d16695e73d925d": "chaque terme se calcule à partir du précédent : moitié, puis plus 3"}, "shape": [3]}
📝 Ex POS.10a : réponse enregistrée (array, 1 décimale(s)).
WBM_ANSWER {"hash": "2410b7efbf8a61584e9d2f9c9ea6651949d142006c147af912087d7f04010194", "id": "POS.10b", "kind": "int", "magnitude": 0, "mistakes": {"2709cd2218d41b39d171c6fed55b72fe5d8bcc387b94cffa5000434e23102142": "la limite vérifie la même relation que les termes de la suite"}, "sign": 1

In [22]:
answer_POS_10a = [4, 5, 5.5]
answer_POS_10b = 6

RESULTS["POS.10"] = check_all("POS.10", "ab", [answer_POS_10a, answer_POS_10b])

POS.10a ✅
POS.10b ✅


## R6 · Exponentielle et logarithmes

**POS.11 — Propriétés de l'exponentielle et du logarithme** · a) un nombre · b) un entier · c) une fraction (entre guillemets) ou un décimal · d) un nombre

In [23]:
wbm.record("POS.11a", 1, mistakes={"e puissance 0 vaut 1, pas 0": 0})
wbm.record("POS.11b", 7, mistakes={"ln a - ln b = ln(a / b), pas ln(a - b)": 18})
wbm.record("POS.11c", "1/2", kind="fraction", mistakes={"la racine carrée est la puissance 1/2 : écris √e comme une puissance de e": "2"})
wbm.record("POS.11d", 6, mistakes={"log₂ 64 est l'exposant de 2 qui donne 64": 32})

WBM_ANSWER {"hash": "863fc48c3a44b210fb5bd02bb872df823ac7ca3c43648740f1431824de1c35eb", "id": "POS.11a", "kind": "int", "magnitude": 0, "mistakes": {"3803a9d1d0ebb23b59d62259a945baf555accb7e0c62895070f291dcab7326a7": "e puissance 0 vaut 1, pas 0"}, "sign": 1}
📝 Ex POS.11a : réponse enregistrée (int).
WBM_ANSWER {"hash": "e3548d91239374a326c58232fbbe23115342ef4cff9f841876a73887480a5ead", "id": "POS.11b", "kind": "int", "magnitude": 0, "mistakes": {"1e5a1bf606c2faa03f21c68ed3a546a12a386c6d4b290a9ce9a7eb4ae26529b7": "ln a - ln b = ln(a / b), pas ln(a - b)"}, "sign": 1}
📝 Ex POS.11b : réponse enregistrée (int).
WBM_ANSWER {"hash": "8ee7c70960d7851564f7be2255404212f4a432147e3532a62f5dc68d969169fb", "id": "POS.11c", "kind": "fraction", "mistakes": {"9e0ea62a9f30b06beb17d8609a5d5fe1d447d36392962d9507bff09b3b926eed": "la racine carrée est la puissance 1/2 : écris √e comme une puissance de e"}, "sign": 1}
📝 Ex POS.11c : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "4fd63e61adf1c257d198f5

In [24]:
answer_POS_11a = 1
answer_POS_11b = 7
answer_POS_11c = "1/2"
answer_POS_11d = 6

RESULTS["POS.11"] = check_all("POS.11", "abcd", [answer_POS_11a, answer_POS_11b, answer_POS_11c, answer_POS_11d])

POS.11a ✅
POS.11b ✅
POS.11c ✅
POS.11d ✅


**POS.12 — Résoudre des équations avec exp et ln** · a) une expression exacte avec `ln`, entre guillemets · b) une expression exacte avec `exp`, entre guillemets · c) un nombre

In [25]:
wbm.record("POS.12a", "ln(7)", kind="expr", variables=("x",), mistakes={"on défait l'exponentielle avec le logarithme népérien, pas avec l'exponentielle": "exp(7)"})
wbm.record("POS.12b", "exp(-1)", kind="expr", variables=("x",), mistakes={"applique exp aux deux membres, sans perdre le signe du second membre": "exp(1)", "une exponentielle est toujours positive": "-exp(1)"})
wbm.record("POS.12c", 2.0, decimals=2, mistakes={"e puissance y vaut 1 seulement quand y vaut 0, pas quand y vaut 1": round(7 / 3, 2)})

WBM_ANSWER {"hash": "1172e198270192b10c70395c4c4a1b1756beb0f2e3ff68669c3c72f7455787e7", "id": "POS.12a", "kind": "expr", "mistakes": {"b8094b905de5ea08ce84933127a7245b0318da851a6a36e3ff0ddf2ba7325a2b": "on défait l'exponentielle avec le logarithme népérien, pas avec l'exponentielle"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex POS.12a : réponse enregistrée (expr).
WBM_ANSWER {"hash": "3b2efd5626039e106b93a82cbe8114712c6521536e8971dc1819a5ccb17f6fbc", "id": "POS.12b", "kind": "expr", "mistakes": {"1539c7ca91f51fed1319e6e1e3861c419c786868c3868108d3df6567bc2fa8e0": "applique exp aux deux membres, sans perdre le signe du second membre", "60b1c1045fcba514f0a2585655a014b602fe18c965b1e5c3515ba16b28c47376": "une exponentielle est toujours positive"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex POS.12b : réponse enregistrée (expr).
WBM_ANSWER {"decimals": 2, "hash": "f857854d55792d2b2d0a90347ee11300902a106634110a54b7

In [26]:
answer_POS_12a = "ln(7)"
answer_POS_12b = "exp(-1)"
answer_POS_12c = 2.0

RESULTS["POS.12"] = check_all("POS.12", "abc", [answer_POS_12a, answer_POS_12b, answer_POS_12c])

POS.12a ✅
POS.12b ✅
POS.12c ✅


## R7 · Limites et dérivation

**POS.13 — Dériver trois fonctions, dont une composée** · a) une expression en x, entre guillemets · b) une expression en x, entre guillemets · c) une expression en x, entre guillemets

In [27]:
wbm.record("POS.13a", "10*x**4 - 6*x", kind="expr", variables=("x",), mistakes={"la dérivée d'une constante est nulle": "10*x**4 - 6*x + 1", "(x^n)' = n x^(n-1) : l'exposant baisse de 1": "10*x**5 - 6*x**2"})
wbm.record("POS.13b", "x*exp(x)", kind="expr", variables=("x",), mistakes={"un produit se dérive par (uv)' = u'v + uv'": "exp(x)"})
wbm.record("POS.13c", "-6*(5 - 2*x)**2", kind="expr", variables=("x",), mistakes={"n'oublie pas la dérivée de l'intérieur (règle de la chaîne)": "3*(5 - 2*x)**2", "la dérivée de l'intérieur, 5 - 2x, est négative": "6*(5 - 2*x)**2"})

WBM_ANSWER {"hash": "76a4e1ede9e1288f032336dd4f02edf1ef1802a3d5518f979c0175a143210489", "id": "POS.13a", "kind": "expr", "mistakes": {"272affb1faf9ced29f0b136b5b0b2390dfc694c34f46967fe4237f839330da24": "la dérivée d'une constante est nulle", "bf8589afb2571e7a32edd96709e46459e2e8e41a3fafb7cd16855b35e2854f16": "(x^n)' = n x^(n-1) : l'exposant baisse de 1"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex POS.13a : réponse enregistrée (expr).
WBM_ANSWER {"hash": "af6cb989c363ec015d994bf38e7ce6cec57232db1a3ad1e40f581392ea31fb8b", "id": "POS.13b", "kind": "expr", "mistakes": {"d75d67f6bdb5bdb1f107f592b8d82f5edf135d512257a7c0680562f93accb303": "un produit se dérive par (uv)' = u'v + uv'"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex POS.13b : réponse enregistrée (expr).


WBM_ANSWER {"hash": "29fcd94babcbcceed858e89da71fd92b54ec1a74c57830cab81205f07b73f7e5", "id": "POS.13c", "kind": "expr", "mistakes": {"c4ebcc30d83f17a64811b86bec69178344da6029417940edc5835b5b086be53c": "n'oublie pas la dérivée de l'intérieur (règle de la chaîne)", "d4880b30dfac1327385188f915052833fe4f889728463cc06f82f812760a0cad": "la dérivée de l'intérieur, 5 - 2x, est négative"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex POS.13c : réponse enregistrée (expr).


In [28]:
answer_POS_13a = "10*x**4 - 6*x"
answer_POS_13b = "x*exp(x)"
answer_POS_13c = "-6*(5 - 2*x)**2"

RESULTS["POS.13"] = check_all("POS.13", "abc", [answer_POS_13a, answer_POS_13b, answer_POS_13c])

POS.13a ✅
POS.13b ✅
POS.13c ✅


**POS.14 — Tangente, variations et une limite** · a) la liste `[m, p]` · b) un intervalle, entre guillemets · c) un nombre · d) un nombre

In [29]:
wbm.record("POS.14a", [9, -16], mistakes={"p n'est pas f(2) : écris y = f'(2)(x - 2) + f(2), puis développe": [9, 2]})
wbm.record("POS.14b", "[-1 ; 1]", kind="interval", mistakes={"le plus grand intervalle contient les bornes où f' s'annule : crochets fermés": "]-1 ; 1["})
wbm.record("POS.14c", 2, mistakes={"le maximum local est une valeur de f(x), pas une abscisse": -1})
wbm.record("POS.14d", 3, mistakes={"en +∞, seuls les termes de plus haut degré comptent, en haut comme en bas": 0})

WBM_ANSWER {"decimals": 0, "element_hashes": ["c2758350613522d27f2f472db719ff1273050b324cbc6e4bab510306f3b8609a", "7912babdfccc60f3415f56be06c7666139d8f43140ba0f7bca6aa866a5c096a4"], "hash": "dcd8e78b49e8083f2291d1ed9e74acba88fcc70c8faf480c53086c3fcc271dfb", "id": "POS.14a", "integer": true, "kind": "array", "mistakes": {"42b22a1f08eb5393ca0ff0bd497ea4a6a9f9da7371ec312fe15f7da706306304": "p n'est pas f(2) : écris y = f'(2)(x - 2) + f(2), puis développe"}, "shape": [2]}
📝 Ex POS.14a : réponse enregistrée (array, 0 décimale(s)).
WBM_ANSWER {"hash": "52e121bac6d6d0a62dc82c0a2baf05b98ad083a3d13ff00ba03ed8227caf41e2", "id": "POS.14b", "kind": "interval", "mistakes": {"dd74ad6bf66e2e07b68baa6668d7dbf80204c94626d25f634296c8d11c737aa4": "le plus grand intervalle contient les bornes où f' s'annule : crochets fermés"}}
📝 Ex POS.14b : réponse enregistrée (interval).
WBM_ANSWER {"hash": "4f8ba6c429ebc40b1b329c150feb89c9e0c767adb0b4eaa8c81a48e65966c89d", "id": "POS.14c", "kind": "int", "magnitude":

In [30]:
answer_POS_14a = [9, -16]
answer_POS_14b = "[-1 ; 1]"
answer_POS_14c = 2
answer_POS_14d = 3

RESULTS["POS.14"] = check_all("POS.14", "abcd", [answer_POS_14a, answer_POS_14b, answer_POS_14c, answer_POS_14d])

POS.14a ✅
POS.14b ✅
POS.14c ✅
POS.14d ✅


## R8 · Probabilités de base

**POS.15 — Une probabilité avec un arbre** · a) une fraction irréductible, entre guillemets · b) une fraction irréductible, entre guillemets

In [31]:
wbm.record("POS.15a", Fraction(3, 5) * Fraction(2, 4), kind="fraction", simplified=True, mistakes={"sans remise, le second tirage se fait avec une boule de moins": "9/25"})
wbm.record("POS.15b", 2 * Fraction(3, 5) * Fraction(2, 4), kind="fraction", simplified=True, mistakes={"une de chaque couleur : rouge puis bleue, ou bleue puis rouge": "3/10"})

WBM_ANSWER {"hash": "eb30075a19a2cd2812760e37dceb851c546621748320aeb0fa8816cf5cb0205c", "id": "POS.15a", "kind": "fraction", "mistakes": {"0979dc33900824c6933e60bc142a3dea2c2e25fc62021a18cd08cc19a62e5531": "sans remise, le second tirage se fait avec une boule de moins"}, "sign": 1, "simplified": true}
📝 Ex POS.15a : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "89a0bd7637e4d3cd7e9018479a31a0de1ea4f20fc0105e1dd922a98c7dd3d32f", "id": "POS.15b", "kind": "fraction", "mistakes": {"549bdbbaf9600b155e768bfd00a7eb2f324ed1402cbf85f537acc45430f21910": "une de chaque couleur : rouge puis bleue, ou bleue puis rouge"}, "sign": 1, "simplified": true}
📝 Ex POS.15b : réponse enregistrée (fraction).


In [32]:
answer_POS_15a = Fraction(3, 5) * Fraction(2, 4)
answer_POS_15b = 2 * Fraction(3, 5) * Fraction(2, 4)

RESULTS["POS.15"] = check_all("POS.15", "ab", [answer_POS_15a, answer_POS_15b])

POS.15a ✅
POS.15b ✅


**POS.16 — L'espérance et la variance d'une variable aléatoire** · a) un décimal, avec un point · b) un décimal, avec un point

In [33]:
wbm.record("POS.16a", 0.3, decimals=2, mistakes={"l'espérance pondère chaque valeur par sa probabilité": round(-1 / 3, 2), "attention au signe de la valeur négative": 0.7})
wbm.record("POS.16b", 0.81, decimals=2, mistakes={"V = E[X²] - E[X]² : soustrais le carré de l'espérance": 0.9, "c'est le carré de l'espérance qu'on soustrait": 0.6})

WBM_ANSWER {"decimals": 2, "hash": "f4c2989eceb9c57a25220f2a34f9f00c7b130db1b3aada3851727c44681f0b65", "hash_coarse": "8c8580d8ddd0e70b2ecfe4b581206750126199fef336611142c66e81ce496e27", "id": "POS.16a", "kind": "float", "magnitude": -1, "mistakes": {"7e1f5146293c108102743d722e79dadaac7fd0f92c4c891da853bd7bf1037d2e": "l'espérance pondère chaque valeur par sa probabilité", "c74a6ef0675ce4ec7eaede342e46db1e0b42af044c9af4ca108daecc931a7547": "attention au signe de la valeur négative"}, "sign": 1}
📝 Ex POS.16a : réponse enregistrée (float, 2 décimale(s)).
WBM_ANSWER {"decimals": 2, "hash": "e8ce03e554438bd35085b5941bcd40816a28dbc5bd96bcd9f6dd241314926922", "hash_coarse": "39b2afd5ca421667abb4a7bbf323732c3fdb257d99e4c1a6f72753c15904160e", "id": "POS.16b", "kind": "float", "magnitude": -1, "mistakes": {"320c3e452a670dc39d16ffccb5f3cc93390529e27ab27a873130143d95fe8df0": "V = E[X²] - E[X]² : soustrais le carré de l'espérance", "c9f44957c83a16747a900518e1e5d970ab8ebafda3eafe453c6c936ddf6f34ba": 

In [34]:
answer_POS_16a = 0.3
answer_POS_16b = 0.81

RESULTS["POS.16"] = check_all("POS.16", "ab", [answer_POS_16a, answer_POS_16b])

POS.16a ✅
POS.16b ✅


## R9 · Géométrie du plan, vecteurs et trigonométrie

**POS.17 — Distance et produit scalaire dans le plan** · a) un nombre · b) un nombre · c) `True` ou `False`

In [35]:
wbm.record("POS.17a", 13, decimals=0, mistakes={"la distance est la racine de la somme des carrés des écarts": 17, "n'oublie pas la racine carrée": 169, "attention au signe : soustraire une ordonnée négative revient à ajouter": 11})
wbm.record("POS.17b", 3 * 4 + (-2) * 1, mistakes={"attention au signe du produit des secondes coordonnées": 14})
wbm.record("POS.17c", 3 * 2 + (-2) * 3 == 0, mistakes={"deux vecteurs sont orthogonaux quand leur produit scalaire est nul : calcule-le": False})

WBM_ANSWER {"hash": "4e6d2f47f5b604929472bf6ed477e81ac51fdd38844c57498412cfcc2fdce681", "id": "POS.17a", "kind": "int", "magnitude": 1, "mistakes": {"4995584364512bc7c5370fa7006edb362cb37d48613f8090cd1c599df416edc8": "n'oublie pas la racine carrée", "921bd2e9c36c02044a9edd5108298bae92b5fb33a408b6762fa7181df5dd7f2a": "la distance est la racine de la somme des carrés des écarts", "9ec4d619e287efde8e40660b8cc22b933f45455313ad88f3d38515f54c7579a5": "attention au signe : soustraire une ordonnée négative revient à ajouter"}, "sign": 1}
📝 Ex POS.17a : réponse enregistrée (int).
WBM_ANSWER {"hash": "47154abe171e750995080c5bd5c09ed33902c1ff392d2d8ec73aba2c7be5646a", "id": "POS.17b", "kind": "int", "magnitude": 1, "mistakes": {"00234e4f5950a36524f9749e767c9e295f1303f63abf02f4f8e67eb448a0df74": "attention au signe du produit des secondes coordonnées"}, "sign": 1}
📝 Ex POS.17b : réponse enregistrée (int).
WBM_ANSWER {"hash": "a586600cef3044cbc5dc6569d8597cebc24384a06b5033233249e38a7e61382e", "id":

In [36]:
answer_POS_17a = 13
answer_POS_17b = 3 * 4 + (-2) * 1
answer_POS_17c = 3 * 2 + (-2) * 3 == 0

RESULTS["POS.17"] = check_all("POS.17", "abc", [answer_POS_17a, answer_POS_17b, answer_POS_17c])

POS.17a ✅
POS.17b ✅
POS.17c ✅


**POS.18 — Cosinus d'un angle et radians** · a) une fraction, entre guillemets · b) une fraction, entre guillemets · c) un nombre (en degrés, sans le signe °)

In [37]:
wbm.record("POS.18a", Fraction(150, 180), kind="fraction", mistakes={"180 degrés correspondent à π radians": "6/5"})
wbm.record("POS.18b", "1/2", kind="fraction")
wbm.record("POS.18c", 60, mistakes={"les vecteurs ne sont pas orthogonaux : calcule le cosinus de leur angle": 90, "cos θ = u·v / (‖u‖ ‖v‖) : l'angle est celui dont le cosinus vaut ce quotient": 30})

WBM_ANSWER {"hash": "0b9b139f1bad37e336c197782795f1d8c3071302389e8b5c3f49b26017ef9e5b", "id": "POS.18a", "kind": "fraction", "mistakes": {"3f6be988ca2a012345b8cdc8ce402fcbc077dbb8d7534fac005764c17caf13e4": "180 degrés correspondent à π radians"}, "sign": 1}
📝 Ex POS.18a : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "58a8f152abb666949b60c0c2ccd43980638613ae21823ef2257e36c7e1397288", "id": "POS.18b", "kind": "fraction", "sign": 1}
📝 Ex POS.18b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "9221c2ed38186f2c8e5ae9121cf8f545b30d7f6ac2f6740a1aa535ff089934b1", "id": "POS.18c", "kind": "int", "magnitude": 1, "mistakes": {"38e364c4a7e79729ec1dc00bb1b69e95d1fd89b8cbf07f71b2de20b372314374": "les vecteurs ne sont pas orthogonaux : calcule le cosinus de leur angle", "84152abe496cc559611efbf3330e9558777160c2d0cf12b00920be79cf494aab": "cos θ = u·v / (‖u‖ ‖v‖) : l'angle est celui dont le cosinus vaut ce quotient"}, "sign": 1}
📝 Ex POS.18c : réponse enregistrée (int).


In [38]:
answer_POS_18a = Fraction(150, 180)
answer_POS_18b = "1/2"
answer_POS_18c = 60

RESULTS["POS.18"] = check_all("POS.18", "abc", [answer_POS_18a, answer_POS_18b, answer_POS_18c])

POS.18a ✅
POS.18b ✅
POS.18c ✅


## Ta grille

Un exercice compte quand **toutes** ses questions sont justes. Deux exercices par chapitre : 2/2 → tu peux sauter, 1/2 → à survoler, 0/2 → à faire en entier. Le détail, et ce qu'il faut faire quand même, est dans `05_grille.md`.

In [39]:
show_grid(RESULTS, FINISHED)

R1 · Nombres et calcul                              2/2 → tu peux sauter
R2 · Calcul littéral et équations                   2/2 → tu peux sauter
R3 · Fonctions                                      2/2 → tu peux sauter
R4 · Logique, ensembles, Σ et Π                     2/2 → tu peux sauter
R5 · Suites                                         2/2 → tu peux sauter
R6 · Exponentielle et logarithmes                   2/2 → tu peux sauter
R7 · Limites et dérivation                          2/2 → tu peux sauter
R8 · Probabilités de base                           2/2 → tu peux sauter
R9 · Géométrie du plan, vecteurs et trigonométrie   2/2 → tu peux sauter

Ce que veut dire chaque verdict, et ce qu'il faut faire quand même : positionnement/05_grille.md


## Et maintenant

Recopie ta grille dans ta copie de `04_mes_reponses.md` et dans `mon_travail/suivi/journal.md`. Lis `03_corrige.md` pour les questions ratées : chacune renvoie à la section du chapitre qui l'enseigne. Puis passe à R1 : même « sauté », un chapitre garde quelques exercices à faire (`05_grille.md`).